<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/Daily_Challenge_MCP_Airbnb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge — MCP + Airbnb Mini-Agent (Completed)

This notebook connects two MCP servers over **STDIO**:

1. A local notes server with `add_note` and `list_notes`.
2. An Airbnb server — a local stub by default, or the real npm server optionally.
3. A stub planner by default, or GitHub Models optionally.

The default configuration requires **no API token** and runs end to end in Google Colab.


## Install
Run once. npm only needed for the real Airbnb server.

In [ ]:
# Core dependencies
%pip install -q mcp nest_asyncio requests azure-ai-inference

# Optional only for the real Airbnb MCP server:
# !npm install -g @openbnb/mcp-server-airbnb


In [ ]:
# Compatibility patch for some Colab/Jupyter subprocess environments
import sys

try:
    from ipykernel.iostream import OutStream

    def _patched_fileno(self):
        return 2 if self is sys.stderr else 1

    OutStream.fileno = _patched_fileno
    sys.stdout.fileno = lambda: 1
    sys.stderr.fileno = lambda: 2
except Exception:
    pass


## Config
Flip toggles as needed. Keep defaults for stubbed run.

In [ ]:
import os
from pathlib import Path

# Optional authentication value. The local STDIO demo does not require it.
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "")

# Safe defaults for the required submission run.
USE_REAL_AIRBNB = False
USE_REAL_LLM = False

print("Airbnb mode:", "real npm server" if USE_REAL_AIRBNB else "local stub")
print("Planner mode:", "GitHub Models" if USE_REAL_LLM else "local stub")


Airbnb mode: local stub
Planner mode: local stub


In [ ]:
import os

BASE_ENV = os.environ.copy()
BASE_ENV["MCP_HTTP_TOKEN"] = MCP_HTTP_TOKEN
BASE_ENV["PYTHONUNBUFFERED"] = "1"


In [ ]:
# Load a GitHub token only when the real LLM mode is enabled.
# In Colab, store it under the secret name GITHUB_TOKEN.
if USE_REAL_LLM:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        if token:
            os.environ["GITHUB_TOKEN"] = token
    except Exception:
        pass

print("GITHUB_TOKEN available:", bool(os.getenv("GITHUB_TOKEN")))


GITHUB_TOKEN available: False


## Local notes MCP server

In [ ]:
LOCAL_SERVER = Path("local_notes_server.py")
LOCAL_SERVER.write_text(
    """
import logging
logging.disable(logging.CRITICAL)

from mcp.server.fastmcp import FastMCP

notes = []
mcp = FastMCP("Local Notes")

@mcp.tool()
def add_note(text: str) -> str:
    \"\"\"Add a note to the in-memory list.\"\"\"
    notes.append(text)
    return f"Saved note #{len(notes)}: {text}"

@mcp.tool()
def list_notes() -> str:
    \"\"\"List all saved notes.\"\"\"
    if not notes:
        return "No notes yet"
    return "\\n".join(f"{i + 1}. {note}" for i, note in enumerate(notes))

if __name__ == "__main__":
    mcp.run()
""".strip() + "\n",
    encoding="utf-8",
)
print("Wrote:", LOCAL_SERVER)


Wrote: local_notes_server.py


## Airbnb stub MCP server

The stub returns deterministic sample listings, so the notebook can be submitted without npm, credentials, or network access.


In [ ]:
AIRBNB_STUB_SERVER = Path("airbnb_stub_server.py")
AIRBNB_STUB_SERVER.write_text(
    """
import logging
logging.disable(logging.CRITICAL)

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Airbnb Stub")

@mcp.tool()
def search_listings(city: str, guests: int = 1) -> str:
    \"\"\"Search deterministic sample Airbnb listings for a city.\"\"\"
    listings = [
        {
            "name": f"Central Studio in {city}",
            "price_per_night": 42000,
            "currency": "XOF",
            "rating": 4.8,
            "max_guests": 2,
        },
        {
            "name": f"Modern Apartment in {city}",
            "price_per_night": 65000,
            "currency": "XOF",
            "rating": 4.9,
            "max_guests": 4,
        },
        {
            "name": f"Cozy Guesthouse in {city}",
            "price_per_night": 35000,
            "currency": "XOF",
            "rating": 4.6,
            "max_guests": 2,
        },
    ]

    eligible = [listing for listing in listings if listing["max_guests"] >= guests]
    if not eligible:
        return f"No sample listing in {city} can host {guests} guests."

    return "\\n".join(
        (
            f'{index}. {listing["name"]} — '
            f'{listing["price_per_night"]:,} {listing["currency"]}/night — '
            f'rating {listing["rating"]}'
        )
        for index, listing in enumerate(eligible, start=1)
    )

if __name__ == "__main__":
    mcp.run()
""".strip() + "\n",
    encoding="utf-8",
)
print("Wrote:", AIRBNB_STUB_SERVER)


Wrote: airbnb_stub_server.py


## Client helpers (convert tools, stub planner, optional real LLM)

In [ ]:
import json
import re
import nest_asyncio
from typing import Any, Dict, List

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


def convert_tool(tool, prefix: str) -> Dict[str, Any]:
    """Convert an MCP tool into an LLM-compatible function specification."""
    function_name = f"{prefix}__{tool.name}"
    input_schema = tool.inputSchema or {}

    return {
        "type": "function",
        "function": {
            "name": function_name,
            "description": tool.description or "MCP tool",
            "parameters": {
                "type": "object",
                "properties": input_schema.get("properties", {}),
                "required": input_schema.get("required", []),
            },
        },
    }


def _extract_city(prompt: str) -> str:
    """Extract a simple city name from the demo prompt."""
    match = re.search(
        r"\b(?:in|for|à|a)\s+([A-Za-zÀ-ÿ][A-Za-zÀ-ÿ' -]{1,40}?)(?=\s+(?:for|pour|with|avec|and|et|,)|[?.!]|$)",
        prompt,
        flags=re.IGNORECASE,
    )
    return match.group(1).strip().rstrip(" ,;:") if match else "Abidjan"


def _extract_guests(prompt: str) -> int:
    match = re.search(r"(\d+)\s+(?:guests?|people|persons?|voyageurs?|personnes?)", prompt, re.IGNORECASE)
    return int(match.group(1)) if match else 1


def _extract_note(prompt: str, city: str) -> str:
    match = re.search(
        r"(?:save|add|write)\s+(?:a\s+)?note(?:\s+that|\s*:)?\s*(.+?)(?=\s+(?:then|and then)\s+list|[.!?]|$)",
        prompt,
        re.IGNORECASE,
    )
    if match:
        return match.group(1).strip().rstrip(" ,;:")
    return f"Compare the Airbnb options found in {city}."


def stub_plan(prompt: str, functions: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Create deterministic tool calls without using an external LLM."""
    available = {item["function"]["name"]: item["function"] for item in functions}
    lower_prompt = prompt.lower()
    calls: List[Dict[str, Any]] = []

    city = _extract_city(prompt)
    guests = _extract_guests(prompt)

    airbnb_search = next(
        (
            name
            for name in available
            if name.startswith("airbnb__") and "search" in name.lower()
        ),
        None,
    )

    if airbnb_search and any(
        word in lower_prompt
        for word in ["airbnb", "listing", "stay", "accommodation", "hébergement", "logement"]
    ):
        properties = available[airbnb_search]["parameters"].get("properties", {})
        args: Dict[str, Any] = {}

        for key in properties:
            key_lower = key.lower()
            if key_lower in {"city", "location", "destination", "query", "place"}:
                args[key] = city
            elif key_lower in {"guests", "adults", "people", "persons"}:
                args[key] = guests

        calls.append({"name": airbnb_search, "args": args})

    notes_add = next(
        (
            name
            for name in available
            if name.startswith("notes__") and "add_note" in name.lower()
        ),
        None,
    )
    if notes_add and "note" in lower_prompt and any(
        word in lower_prompt for word in ["save", "add", "write"]
    ):
        calls.append(
            {
                "name": notes_add,
                "args": {"text": _extract_note(prompt, city)},
            }
        )

    notes_list = next(
        (
            name
            for name in available
            if name.startswith("notes__") and "list_notes" in name.lower()
        ),
        None,
    )
    if notes_list and any(
        phrase in lower_prompt
        for phrase in ["list my notes", "show my notes", "list notes", "show notes"]
    ):
        calls.append({"name": notes_list, "args": {}})

    return calls


def call_llm(
    prompt: str,
    functions: List[Dict[str, Any]],
    use_real: bool = False,
) -> List[Dict[str, Any]]:
    """Use the local planner by default, or GitHub Models when enabled."""
    if not use_real:
        return stub_plan(prompt, functions)

    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or keep USE_REAL_LLM=False.")

    client = ChatCompletionsClient(
        endpoint="https://models.inference.ai.azure.com",
        credential=AzureKeyCredential(token),
    )

    response = client.complete(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an MCP tool planner. Select only the tools needed "
                    "to satisfy the user's request."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        tools=functions,
        tool_choice="auto",
        temperature=0,
        max_tokens=400,
    )

    calls: List[Dict[str, Any]] = []
    message = response.choices[0].message

    for tool_call in message.tool_calls or []:
        arguments = tool_call.function.arguments
        parsed_args = json.loads(arguments) if isinstance(arguments, str) else arguments
        calls.append(
            {
                "name": tool_call.function.name,
                "args": parsed_args or {},
            }
        )

    return calls


In [ ]:
def answer_with_llm(
    user_prompt: str,
    tool_calls: List[Dict[str, Any]],
    tool_results: List[Dict[str, Any]],
    use_real: bool = False,
) -> str:
    """Format a final answer locally, or use GitHub Models when enabled."""
    if not use_real:
        lines = ["## Mini-agent result", ""]
        for result in tool_results:
            lines.append(f"### `{result['name']}`")
            content = result.get("content", [])
            lines.append("\n".join(content) if content else "No textual output.")
            lines.append("")

        distinct_tools = list(dict.fromkeys(call["name"] for call in tool_calls))
        lines.append("## Tools used")
        if distinct_tools:
            lines.extend(f"- `{name}`" for name in distinct_tools)
        else:
            lines.append("- None")
        return "\n".join(lines)

    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or keep USE_REAL_LLM=False.")

    client = ChatCompletionsClient(
        endpoint="https://models.inference.ai.azure.com",
        credential=AzureKeyCredential(token),
    )

    compact_results = []
    for result in tool_results:
        content = result.get("content", [])
        compact_results.append(
            {
                "name": result.get("name"),
                "args": result.get("args", {}),
                "content": [
                    item[:4000] + ("...(truncated)" if len(item) > 4000 else "")
                    for item in content[:2]
                ],
            }
        )

    payload = {
        "user_question": user_prompt,
        "tool_calls": tool_calls,
        "tool_results": compact_results,
    }

    response = client.complete(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer clearly using only the supplied tool outputs. "
                    "End with a '## Tools used' section."
                ),
            },
            {
                "role": "user",
                "content": json.dumps(payload, ensure_ascii=False),
            },
        ],
        temperature=0,
        max_tokens=600,
    )

    return str(response.choices[0].message.content)


## Orchestrate (connect both servers and execute tool_calls)

In [ ]:
import sys

async def orchestrate(prompt: str):
    """Connect to both MCP servers, discover tools, plan, execute, and return results."""
    local_params = StdioServerParameters(
        command=sys.executable,
        args=[str(LOCAL_SERVER)],
        env=BASE_ENV,
    )

    if USE_REAL_AIRBNB:
        airbnb_params = StdioServerParameters(
            command="npx",
            args=["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"],
            env=BASE_ENV,
        )
    else:
        airbnb_params = StdioServerParameters(
            command=sys.executable,
            args=[str(AIRBNB_STUB_SERVER)],
            env=BASE_ENV,
        )

    async with stdio_client(local_params) as (local_read, local_write):
        async with ClientSession(local_read, local_write) as local_session:
            await local_session.initialize()
            local_tools_response = await local_session.list_tools()

            async with stdio_client(airbnb_params) as (airbnb_read, airbnb_write):
                async with ClientSession(airbnb_read, airbnb_write) as airbnb_session:
                    await airbnb_session.initialize()
                    airbnb_tools_response = await airbnb_session.list_tools()

                    print(
                        "Notes tools:",
                        [tool.name for tool in local_tools_response.tools],
                    )
                    print(
                        "Airbnb tools:",
                        [tool.name for tool in airbnb_tools_response.tools],
                    )

                    functions = (
                        [
                            convert_tool(tool, "notes")
                            for tool in local_tools_response.tools
                        ]
                        + [
                            convert_tool(tool, "airbnb")
                            for tool in airbnb_tools_response.tools
                        ]
                    )

                    print(
                        "LLM function names:",
                        [item["function"]["name"] for item in functions],
                    )

                    tool_calls = call_llm(
                        prompt,
                        functions,
                        use_real=USE_REAL_LLM,
                    )
                    print("Tool calls:")
                    print(json.dumps(tool_calls, indent=2, ensure_ascii=False))

                    tool_results: List[Dict[str, Any]] = []

                    for call in tool_calls:
                        prefixed_name = call["name"]
                        arguments = call.get("args", {})
                        prefix, tool_name = prefixed_name.split("__", 1)

                        if prefix == "notes":
                            result = await local_session.call_tool(tool_name, arguments)
                        elif prefix == "airbnb":
                            result = await airbnb_session.call_tool(tool_name, arguments)
                        else:
                            raise ValueError(f"Unknown tool prefix: {prefix}")

                        text_content = [
                            item.text
                            for item in result.content
                            if hasattr(item, "text")
                        ]

                        tool_results.append(
                            {
                                "name": prefixed_name,
                                "args": arguments,
                                "content": text_content,
                            }
                        )

                    return tool_calls, tool_results


## Demo

The prompt below deliberately triggers both MCP servers:

- Airbnb listing search
- Note creation
- Note listing

Change the city or note text to test another request.


In [ ]:
from IPython.display import Markdown, display

prompt = (
    "Find Airbnb listings in Abidjan for 2 guests, "
    "save a note that I should compare the cheapest options, "
    "then list my notes."
)

tool_calls, tool_results = await orchestrate(prompt)

print("\nRaw tool results:")
print(json.dumps(tool_results, indent=2, ensure_ascii=False))

final_answer = answer_with_llm(
    prompt,
    tool_calls,
    tool_results,
    use_real=USE_REAL_LLM,
)
display(Markdown(final_answer))


Notes tools: ['add_note', 'list_notes']
Airbnb tools: ['search_listings']
LLM function names: ['notes__add_note', 'notes__list_notes', 'airbnb__search_listings']
Tool calls:
[
  {
    "name": "airbnb__search_listings",
    "args": {
      "city": "Abidjan",
      "guests": 2
    }
  },
  {
    "name": "notes__add_note",
    "args": {
      "text": "I should compare the cheapest options"
    }
  },
  {
    "name": "notes__list_notes",
    "args": {}
  }
]

Raw tool results:
[
  {
    "name": "airbnb__search_listings",
    "args": {
      "city": "Abidjan",
      "guests": 2
    },
    "content": [
      "1. Central Studio in Abidjan — 42,000 XOF/night — rating 4.8\n2. Modern Apartment in Abidjan — 65,000 XOF/night — rating 4.9\n3. Cozy Guesthouse in Abidjan — 35,000 XOF/night — rating 4.6"
    ]
  },
  {
    "name": "notes__add_note",
    "args": {
      "text": "I should compare the cheapest options"
    },
    "content": [
      "Saved note #1: I should compare the cheapest options"
 

## Mini-agent result

### `airbnb__search_listings`
1. Central Studio in Abidjan — 42,000 XOF/night — rating 4.8
2. Modern Apartment in Abidjan — 65,000 XOF/night — rating 4.9
3. Cozy Guesthouse in Abidjan — 35,000 XOF/night — rating 4.6

### `notes__add_note`
Saved note #1: I should compare the cheapest options

### `notes__list_notes`
1. I should compare the cheapest options

## Tools used
- `airbnb__search_listings`
- `notes__add_note`
- `notes__list_notes`


## Validation checklist

- Connected to the local notes MCP server over STDIO.
- Connected to the Airbnb stub MCP server over STDIO.
- Discovered and converted MCP tool schemas.
- Planned multiple prefixed tool calls.
- Routed calls to the correct MCP server.
- Printed Airbnb listings and a saved note.
- Default run requires no token.
